In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, precision_recall_curve
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib

In [2]:
# --- Chargement des données ---
df = pd.read_csv('data/creditcard.csv')
 
print("Dimensions du dataset :", df.shape)
print("\nAperçu des colonnes :")
print(df.columns.tolist())
 
# On regarde les premières lignes pour se faire une idée
df.head()

Dimensions du dataset : (284807, 31)

Aperçu des colonnes :
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
# --- Vérifier qu'il n'y a pas de valeurs manquantes ---
# C'est un réflexe systématique : un modèle plante souvent sur des NaN.
print("Valeurs manquantes par colonne :")
print(df.isnull().sum().sum(), "valeurs manquantes au total")

Valeurs manquantes par colonne :
0 valeurs manquantes au total


In [4]:
# --- Le point le plus important : le déséquilibre des classes ---
# 'Class' = 0 pour une transaction normale, 1 pour une fraude
counts = df['Class'].value_counts()
percentages = df['Class'].value_counts(normalize=True) * 100
 
print("\nRépartition des classes :")
print(counts)
print("\nEn pourcentage :")
print(percentages.round(4))
 
# Tu devrais voir quelque chose comme :
# 0 (normal)  : 284315  (~99.83%)
# 1 (fraude)  :    492  (~0.17%)
# C'est ÉNORME comme déséquilibre. Retiens ce chiffre, il justifie
# tout ce qu'on va faire ensuite (métriques adaptées, gestion du
# déséquilibre dans le modèle, etc.)


Répartition des classes :
Class
0    284315
1       492
Name: count, dtype: int64

En pourcentage :
Class
0    99.8273
1     0.1727
Name: proportion, dtype: float64


In [5]:
# --- Regarder la colonne Amount (montant de la transaction) ---
print("\nStatistiques sur les montants (Amount) :")
print(df['Amount'].describe())
 
# Comparons les montants moyens fraude vs non-fraude
print("\nMontant moyen par classe :")
print(df.groupby('Class')['Amount'].mean())
 
# On regarde souvent si les fraudes ont des montants différents
# des transactions normales (parfois oui, parfois non — à toi
# de voir sur TES résultats, ne suppose rien à l'avance)


Statistiques sur les montants (Amount) :
count    284807.000000
mean         88.349619
std         250.120109
min           0.000000
25%           5.600000
50%          22.000000
75%          77.165000
max       25691.160000
Name: Amount, dtype: float64

Montant moyen par classe :
Class
0     88.291022
1    122.211321
Name: Amount, dtype: float64


In [6]:
# --- Vérifier la colonne Time ---
# Time = nombre de secondes écoulées depuis la première transaction
# du dataset. Utile si on veut plus tard simuler un ordre chronologique.
print("\nDurée totale couverte par le dataset (en heures) :")
print(df['Time'].max() / 3600)


Durée totale couverte par le dataset (en heures) :
47.99777777777778


In [7]:
# --- Séparer features (X) et cible (y) ---
X = df.drop(columns=['Class'])
y = df['Class']

In [8]:
# --- Split stratifié ---
# stratify=y est CRUCIAL ici : sans lui, un split aléatoire classique
# pourrait par malchance mettre très peu (ou beaucoup) de fraudes dans
# le test set, faussant complètement l'évaluation.
# test_size=0.2 → 20% des données pour le test, 80% pour l'entraînement.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42  # fixe la "graine" aléatoire pour des résultats reproductibles
)
 
print("Taille train :", X_train.shape)
print("Taille test :", X_test.shape)
 
print("\nProportion de fraudes dans train :", y_train.mean())
print("Proportion de fraudes dans test :", y_test.mean())
# Ces deux proportions doivent être quasi identiques (~0.00173) —
# c'est exactement ce que stratify=y garantit.

Taille train : (227845, 30)
Taille test : (56962, 30)

Proportion de fraudes dans train : 0.001729245759178389
Proportion de fraudes dans test : 0.0017204452090867595


In [9]:
# --- Mise à l'échelle de la colonne Amount ---
# Les colonnes V1-V28 sont déjà issues d'une PCA (donc déjà à une échelle
# comparable). Amount, en revanche, va de 0 à 25 691 — il faut le
# standardiser pour que les modèles sensibles à l'échelle (comme la
# régression logistique) ne soient pas biaisés par cette colonne.
#
# IMPORTANT : on "fit" le scaler UNIQUEMENT sur le train, puis on
# l'applique (transform) sur train ET test. Ne jamais fit sur le test :
# ce serait laisser filtrer de l'information du test set vers
# l'entraînement (on appelle ça une "fuite de données" / data leakage).
scaler = StandardScaler()
 
X_train['Amount'] = scaler.fit_transform(X_train[['Amount']])
X_test['Amount'] = scaler.transform(X_test[['Amount']])
 
print("\nAmount après mise à l'échelle (train) :")
print(X_train['Amount'].describe())
 
# On peut aussi supprimer Time pour ce premier modèle baseline —
# on la réintroduira plus tard si on veut tester un split chronologique.
X_train = X_train.drop(columns=['Time'])
X_test = X_test.drop(columns=['Time'])
 
print("\nColonnes finales utilisées pour l'entraînement :")
print(X_train.columns.tolist())


Amount après mise à l'échelle (train) :
count    2.278450e+05
mean     3.742243e-17
std      1.000002e+00
min     -3.516894e-01
25%     -3.291944e-01
50%     -2.639429e-01
75%     -4.262209e-02
max      1.021170e+02
Name: Amount, dtype: float64

Colonnes finales utilisées pour l'entraînement :
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']


In [10]:
# --- Entraîner le modèle ---
# class_weight='balanced' : le modèle pénalise davantage les erreurs
# sur la classe minoritaire (fraude). Sans ça, avec 0.17% de fraudes,
# le modèle aurait tendance à quasiment ignorer cette classe pour
# minimiser l'erreur globale.
model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,   # la régression logistique a parfois besoin de
                     # plus d'itérations pour converger sur ce dataset
    random_state=42
)
 
model.fit(X_train, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to u

In [11]:
# --- Prédire des PROBABILITÉS, pas juste 0/1 ---
# predict_proba donne la probabilité que chaque transaction soit une
# fraude. C'est ce dont on a besoin pour calculer l'AUC-PR, qui
# évalue le modèle à tous les seuils possibles, pas à un seuil fixe.
y_proba = model.predict_proba(X_test)[:, 1]  # colonne 1 = proba classe "fraude"

In [12]:
# --- Métrique principale : AUC-PR ---
auc_pr = average_precision_score(y_test, y_proba)
print(f"AUC-PR : {auc_pr:.4f}")
# Rappel de l'objectif du cahier des charges : AUC-PR > 0.80

AUC-PR : 0.7199


In [13]:
# --- Regarder aussi les prédictions à un seuil classique de 0.5 ---
y_pred = model.predict(X_test)
 
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))
# Lecture de la matrice :
# [[Vrais négatifs,  Faux positifs],
#  [Faux négatifs,   Vrais positifs]]
# Faux négatif = une fraude qu'on a laissée passer (le pire cas ici)
# Faux positif = une transaction normale bloquée par erreur
 
print("\nRapport de classification :")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraude']))


Matrice de confusion :
[[55425  1439]
 [    8    90]]

Rapport de classification :
              precision    recall  f1-score   support

      Normal       1.00      0.97      0.99     56864
      Fraude       0.06      0.92      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.97      0.99     56962



In [14]:
# --- Calculer precision et recall pour TOUS les seuils possibles ---
# precision_recall_curve teste plein de seuils différents et renvoie,
# pour chacun, la precision et le recall obtenus.
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
 
# precisions et recalls ont un élément de plus que thresholds
# (le dernier point correspond à seuil=1.0, où recall=0)
# on les aligne pour pouvoir les comparer facilement
precisions = precisions[:-1]
recalls = recalls[:-1]

In [15]:
# --- Regarder quelques seuils précis pour comprendre le compromis ---
print("Seuil  | Precision | Recall")
print("-" * 35)
for seuil_cible in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    idx = np.argmin(np.abs(thresholds - seuil_cible))
    print(f"{thresholds[idx]:.2f}   | {precisions[idx]:.3f}     | {recalls[idx]:.3f}")

Seuil  | Precision | Recall
-----------------------------------
0.10   | 0.008     | 0.949
0.20   | 0.016     | 0.939
0.30   | 0.027     | 0.918
0.40   | 0.041     | 0.918
0.50   | 0.059     | 0.918
0.60   | 0.082     | 0.908
0.70   | 0.115     | 0.898
0.80   | 0.165     | 0.898
0.90   | 0.249     | 0.888


In [16]:
# --- Trouver le seuil qui maximise le F1-score (compromis équilibré) ---
# F1 = moyenne harmonique de precision et recall.
# C'est un bon point de départ si on n'a pas de préférence business
# claire entre "rater des fraudes" et "bloquer des clients honnêtes".
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
meilleur_idx = np.argmax(f1_scores)
meilleur_seuil = thresholds[meilleur_idx]
 
print(f"\nMeilleur seuil (F1 max) : {meilleur_seuil:.3f}")
print(f"→ Precision : {precisions[meilleur_idx]:.3f}")
print(f"→ Recall    : {recalls[meilleur_idx]:.3f}")
print(f"→ F1-score  : {f1_scores[meilleur_idx]:.3f}")


Meilleur seuil (F1 max) : 1.000
→ Precision : 0.833
→ Recall    : 0.816
→ F1-score  : 0.825


In [17]:
y_pred_ajuste = (y_proba >= meilleur_seuil).astype(int)
 
print("\nMatrice de confusion avec le seuil ajusté :")
print(confusion_matrix(y_test, y_pred_ajuste))
 
print("\nRapport de classification avec le seuil ajusté :")
print(classification_report(y_test, y_pred_ajuste, target_names=['Normal', 'Fraude']))
 
# Note importante : l'AUC-PR, elle, ne change PAS avec le seuil —
# c'est une métrique qui résume la performance du modèle sur TOUS
# les seuils à la fois. Ajuster le seuil ne change pas la qualité
# du modèle, seulement où on "coupe" pour décider oui/non.


Matrice de confusion avec le seuil ajusté :
[[56848    16]
 [   18    80]]

Rapport de classification avec le seuil ajusté :
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
      Fraude       0.83      0.82      0.82        98

    accuracy                           1.00     56962
   macro avg       0.92      0.91      0.91     56962
weighted avg       1.00      1.00      1.00     56962



In [18]:
# --- Calculer scale_pos_weight ---
# C'est l'équivalent XGBoost de class_weight='balanced'.
# Formule standard : nombre de négatifs / nombre de positifs.
# Ça dit au modèle : "une erreur sur une fraude coûte X fois plus
# cher qu'une erreur sur une transaction normale".
n_negatifs = (y_train == 0).sum()
n_positifs = (y_train == 1).sum()
scale_pos_weight = n_negatifs / n_positifs
 
print(f"scale_pos_weight calculé : {scale_pos_weight:.1f}")
# Avec ~0.17% de fraudes, attends-toi à une valeur autour de 577-580

scale_pos_weight calculé : 577.3


In [19]:
# --- Entraîner XGBoost ---
model_xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',   # on dit explicitement à XGBoost d'optimiser
                            # pour l'AUC-PR, cohérent avec notre métrique cible
    random_state=42,
    n_estimators=200,      # nombre d'arbres ; on ajustera plus tard si besoin
    max_depth=5,           # profondeur max de chaque arbre (évite le sur-apprentissage
                            # si trop grand, vu qu'on n'a que 28 features)
)
 
model_xgb.fit(X_train, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'aucpr'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [20]:
# --- Évaluer avec l'AUC-PR (comme pour la régression logistique) ---
y_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]
 
auc_pr_xgb = average_precision_score(y_test, y_proba_xgb)
print(f"\nAUC-PR (XGBoost) : {auc_pr_xgb:.4f}")
print(f"AUC-PR (régression logistique, pour rappel) : 0.7199")


AUC-PR (XGBoost) : 0.8834
AUC-PR (régression logistique, pour rappel) : 0.7199


In [21]:
# --- Matrice de confusion au seuil par défaut (0.5) ---
y_pred_xgb = model_xgb.predict(X_test)
 
print("\nMatrice de confusion (seuil 0.5) :")
print(confusion_matrix(y_test, y_pred_xgb))
 
print("\nRapport de classification (seuil 0.5) :")
print(classification_report(y_test, y_pred_xgb, target_names=['Normal', 'Fraude']))
 
# On appliquera un ajustement de seuil comme avant une fois qu'on aura
# vu ces premiers résultats — pas la peine de le faire en aveugle.


Matrice de confusion (seuil 0.5) :
[[56854    10]
 [   17    81]]

Rapport de classification (seuil 0.5) :
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
      Fraude       0.89      0.83      0.86        98

    accuracy                           1.00     56962
   macro avg       0.94      0.91      0.93     56962
weighted avg       1.00      1.00      1.00     56962



In [22]:
# --- Appliquer SMOTE UNIQUEMENT sur le train set ---
# Règle d'or, comme pour le scaler : on ne touche JAMAIS au test set.
# Si on appliquait SMOTE avant le split (ou sur le test), on créerait
# des exemples synthétiques qui "fuient" de l'information vers
# l'évaluation, faussant complètement les résultats (data leakage).
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
 
print("Taille du train AVANT SMOTE :", X_train.shape)
print("Taille du train APRÈS SMOTE :", X_train_smote.shape)
 
print("\nRépartition des classes avant SMOTE :")
print(y_train.value_counts())
print("\nRépartition des classes après SMOTE :")
print(y_train_smote.value_counts())
# SMOTE va égaliser les deux classes à 50/50 par défaut

Taille du train AVANT SMOTE : (227845, 29)
Taille du train APRÈS SMOTE : (454902, 29)

Répartition des classes avant SMOTE :
Class
0    227451
1       394
Name: count, dtype: int64

Répartition des classes après SMOTE :
Class
0    227451
1    227451
Name: count, dtype: int64


In [23]:
# --- Entraîner XGBoost sur les données rééquilibrées ---
# Note : on n'utilise PAS scale_pos_weight ici, puisque SMOTE a déjà
# rééquilibré les classes en amont. Les combiner reviendrait à
# sur-corriger le déséquilibre.
model_xgb_smote = XGBClassifier(
    eval_metric='aucpr',
    random_state=42,
    n_estimators=200,
    max_depth=5,
)
 
model_xgb_smote.fit(X_train_smote, y_train_smote)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'aucpr'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [24]:
# --- Évaluer sur le VRAI test set (jamais transformé par SMOTE) ---
y_proba_smote = model_xgb_smote.predict_proba(X_test)[:, 1]
 
auc_pr_smote = average_precision_score(y_test, y_proba_smote)
print(f"\nAUC-PR (XGBoost + SMOTE) : {auc_pr_smote:.4f}")
print("AUC-PR (XGBoost + scale_pos_weight, pour rappel) : 0.8834")
 
y_pred_smote = model_xgb_smote.predict(X_test)
 
print("\nMatrice de confusion (XGBoost + SMOTE, seuil 0.5) :")
print(confusion_matrix(y_test, y_pred_smote))
 
print("\nRapport de classification (XGBoost + SMOTE, seuil 0.5) :")
print(classification_report(y_test, y_pred_smote, target_names=['Normal', 'Fraude']))


AUC-PR (XGBoost + SMOTE) : 0.8610
AUC-PR (XGBoost + scale_pos_weight, pour rappel) : 0.8834

Matrice de confusion (XGBoost + SMOTE, seuil 0.5) :
[[56839    25]
 [   16    82]]

Rapport de classification (XGBoost + SMOTE, seuil 0.5) :
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
      Fraude       0.77      0.84      0.80        98

    accuracy                           1.00     56962
   macro avg       0.88      0.92      0.90     56962
weighted avg       1.00      1.00      1.00     56962



In [25]:
# --- Calculer precision/recall pour tous les seuils ---
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_xgb)
precisions = precisions[:-1]
recalls = recalls[:-1]

In [26]:
# --- Explorer quelques seuils ---
print("Seuil  | Precision | Recall")
print("-" * 35)
for seuil_cible in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    idx = np.argmin(np.abs(thresholds - seuil_cible))
    print(f"{thresholds[idx]:.2f}   | {precisions[idx]:.3f}     | {recalls[idx]:.3f}")

Seuil  | Precision | Recall
-----------------------------------
0.10   | 0.857     | 0.857
0.17   | 0.866     | 0.857
0.31   | 0.884     | 0.857
0.38   | 0.880     | 0.827
0.54   | 0.890     | 0.827
0.57   | 0.900     | 0.827
0.71   | 0.920     | 0.827
0.83   | 0.920     | 0.816
0.91   | 0.930     | 0.816


In [27]:
# --- Meilleur seuil selon F1-score ---
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
meilleur_idx = np.argmax(f1_scores)
meilleur_seuil = thresholds[meilleur_idx]
 
print(f"\nMeilleur seuil (F1 max) : {meilleur_seuil:.3f}")
print(f"→ Precision : {precisions[meilleur_idx]:.3f}")
print(f"→ Recall    : {recalls[meilleur_idx]:.3f}")
print(f"→ F1-score  : {f1_scores[meilleur_idx]:.3f}")


Meilleur seuil (F1 max) : 0.981
→ Precision : 0.988
→ Recall    : 0.806
→ F1-score  : 0.888


In [28]:
# --- Appliquer ce seuil ---
y_pred_ajuste_xgb = (y_proba_xgb >= meilleur_seuil).astype(int)
 
print("\nMatrice de confusion avec le seuil ajusté :")
print(confusion_matrix(y_test, y_pred_ajuste_xgb))
 
print("\nRapport de classification avec le seuil ajusté :")
print(classification_report(y_test, y_pred_ajuste_xgb, target_names=['Normal', 'Fraude']))
 
print("\nPour rappel, résultats au seuil par défaut (0.5) :")
print("Precision : 0.89 | Recall : 0.83")


Matrice de confusion avec le seuil ajusté :
[[56863     1]
 [   19    79]]

Rapport de classification avec le seuil ajusté :
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
      Fraude       0.99      0.81      0.89        98

    accuracy                           1.00     56962
   macro avg       0.99      0.90      0.94     56962
weighted avg       1.00      1.00      1.00     56962


Pour rappel, résultats au seuil par défaut (0.5) :
Precision : 0.89 | Recall : 0.83


In [29]:
# --- Bonus : essayer aussi un seuil orienté "recall" ---
# En fraude, on privilégie parfois volontairement le recall (ne pas
# rater de fraude) au détriment de la precision (accepter plus de
# faux positifs), car le coût d'une fraude non détectée est souvent
# supérieur au coût de vérifier une transaction légitime.
# Ici on cherche le seuil le plus bas qui garde recall >= 0.90
seuils_recall_eleve = [(t, p, r) for t, p, r in zip(thresholds, precisions, recalls) if r >= 0.90]
if seuils_recall_eleve:
    # on prend celui avec la meilleure precision parmi ceux-là
    meilleur_recall_eleve = max(seuils_recall_eleve, key=lambda x: x[1])
    print(f"\nSeuil pour recall >= 0.90 (meilleure precision associée) :")
    print(f"Seuil : {meilleur_recall_eleve[0]:.3f} | Precision : {meilleur_recall_eleve[1]:.3f} | Recall : {meilleur_recall_eleve[2]:.3f}")
else:
    print("\nAucun seuil ne permet d'atteindre un recall >= 0.90 sans effondrer la precision.")


Seuil pour recall >= 0.90 (meilleure precision associée) :
Seuil : 0.001 | Precision : 0.306 | Recall : 0.908


In [30]:
# --- Recharger et préparer les données ---
df = pd.read_csv('data/creditcard.csv')
X = df.drop(columns=['Class'])
y = df['Class']
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
 
scaler = StandardScaler()
X_train['Amount'] = scaler.fit_transform(X_train[['Amount']])
X_test['Amount'] = scaler.transform(X_test[['Amount']])
X_train = X_train.drop(columns=['Time'])
X_test = X_test.drop(columns=['Time'])


In [31]:
# --- Entraîner le modèle final ---
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
 
model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42,
    n_estimators=200,
    max_depth=5,
)
model.fit(X_train, y_train)
 
# Vérification rapide qu'on retrouve bien l'AUC-PR attendu
y_proba = model.predict_proba(X_test)[:, 1]
auc_pr = average_precision_score(y_test, y_proba)
print(f"AUC-PR du modèle sauvegardé : {auc_pr:.4f} (doit être ~0.8834)")

AUC-PR du modèle sauvegardé : 0.8834 (doit être ~0.8834)


In [32]:
# --- Sauvegarder modèle ET scaler ---
# joblib est l'outil standard pour sauvegarder des objets scikit-learn.
# On sauvegarde le scaler AUSSI, car l'API devra appliquer la même
# transformation sur Amount que celle utilisée à l'entraînement.
joblib.dump(model, 'model.joblib')
joblib.dump(scaler, 'scaler.joblib')

['scaler.joblib']

In [33]:
# --- Sauvegarder aussi la liste des colonnes attendues, dans l'ordre ---
# Important : l'API devra recevoir les features exactement dans le
# même ordre que pendant l'entraînement, sinon les prédictions seraient
# fausses sans qu'aucune erreur ne soit levée (piège classique).
feature_columns = X_train.columns.tolist()
joblib.dump(feature_columns, 'feature_columns.joblib')
 
print("\nFichiers sauvegardés :")
print("- model.joblib")
print("- scaler.joblib")
print("- feature_columns.joblib")
print(f"\nColonnes attendues par le modèle ({len(feature_columns)}) :")
print(feature_columns)


Fichiers sauvegardés :
- model.joblib
- scaler.joblib
- feature_columns.joblib

Colonnes attendues par le modèle (29) :
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']


In [34]:
# --- Calculer automatiquement le seuil optimal (F1 max) ---
# Plutôt que de recopier un nombre à la main (source d'erreur si tu
# reraines le modèle et que le seuil change légèrement), on le
# recalcule ici directement à partir des prédictions sur le test set.

precisions, recalls, thresholds_pr = precision_recall_curve(y_test, y_proba)
precisions = precisions[:-1]
recalls = recalls[:-1]
 
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
meilleur_idx = np.argmax(f1_scores)
THRESHOLD = float(thresholds_pr[meilleur_idx])
 
print(f"\nSeuil optimal calculé (F1 max) : {THRESHOLD:.4f}")
print(f"→ Precision : {precisions[meilleur_idx]:.3f}")
print(f"→ Recall    : {recalls[meilleur_idx]:.3f}")
 
joblib.dump(THRESHOLD, 'threshold.joblib')
print(f"\nSeuil de décision sauvegardé : {THRESHOLD:.4f}")


Seuil optimal calculé (F1 max) : 0.9809
→ Precision : 0.988
→ Recall    : 0.806

Seuil de décision sauvegardé : 0.9809
